In [30]:
# Install Libraries
!pip install pandas scikit-learn nltk streamlit pyngrok
import pandas as pd
import numpy as np
import pickle
import re
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score
from sklearn.linear_model import LogisticRegression

In [33]:
# Load Dataset
from google.colab import files
uploaded = files.upload()
df = pd.read_csv("Spam.csv", encoding="latin-1")
df.head()

Saving Spam.csv to Spam.csv


,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN


In [34]:
# Rename columns
df = df[['v1','v2']]
df.columns = ['label','message']

In [35]:
# Encode Target
df['label'] = df['label'].map({
    'ham':0,
    'spam':1
})

In [36]:
# Text Cleaning Function
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z]', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text

In [37]:
# Apply cleaning
df['message'] = df['message'].apply(clean_text)

In [38]:
# Feature Engineering (TF-IDF)
vectorizer = TfidfVectorizer(stop_words='english')
X = vectorizer.fit_transform(df['message'])
y = df['label']

In [39]:
# Train/Test Split
X_train,X_test,y_train,y_test = train_test_split(
    X,y,test_size=0.2,random_state=42
)

In [50]:
# Train Model
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score
vectorizer = TfidfVectorizer(stop_words="english")
X = vectorizer.fit_transform(df["message"])
y = df["label"]
X_train,X_test,y_train,y_test = train_test_split(
    X,y,test_size=0.2,random_state=42
)
model = MultinomialNB()
model.fit(X_train,y_train)
pred = model.predict(X_test)
print("Accuracy:",accuracy_score(y_test,pred))

Accuracy: 0.967713004484305


In [51]:
# Save Model
import pickle
pickle.dump(model, open("spam_model.pkl","wb"))
pickle.dump(vectorizer, open("vectorizer.pkl","wb"))

In [52]:
# Streamlit UI
%%writefile app.py
import streamlit as st
import pickle
import re

# Load model and vectorizer
model = pickle.load(open("spam_model.pkl","rb"))
vectorizer = pickle.load(open("vectorizer.pkl","rb"))
st.set_page_config(
    page_title="AI Spam Detector",
    page_icon="📧",
    layout="wide"
)

# Custom CSS
st.markdown("""
<style>
.title {
font-size:65px;
font-weight:800;
text-align:center;
color:#1F618D;
}
.subtitle {
text-align:center;
font-size:22px;
color:gray;
margin-bottom:20px;
}
.result-box {
padding:20px;
border-radius:12px;
font-size:20px;
text-align:center;
}
</style>
""", unsafe_allow_html=True)

# Header
st.markdown('<p class="title">📧 AI Email Spam Detection</p>', unsafe_allow_html=True)
st.markdown('<p class="subtitle">Detect spam emails using Machine Learning</p>', unsafe_allow_html=True)
st.divider()

# Input Section
st.markdown("### ✉ Enter Email Message")
message = st.text_area(
    "Paste the email content below",
    height=150
)

# Prediction
if st.button("🔍 Analyze Email"):
    if message.strip() == "":
        st.warning("⚠ Please enter an email message first.")
    else:

        # Clean text
        cleaned = re.sub(r'[^a-zA-Z]', ' ', message.lower())
        cleaned = re.sub(r'\s+', ' ', cleaned)

        # Vectorize
        vector = vectorizer.transform([cleaned])

        # Predict
        prediction = model.predict(vector)[0]

        # Probability
        prob = model.predict_proba(vector)[0][prediction]
        st.divider()
        if prediction == 1:
            st.error(f"🚨 This message is **SPAM** (Confidence: {prob*100:.2f}%)")
        else:
            st.success(f"✅ This message is **NOT Spam** (Confidence: {prob*100:.2f}%)")
st.divider()

# Model Information
st.markdown("""
### 🤖 Model Details

**Model Used:** Multinomial Naive Bayes

**Text Processing:**
- Lowercasing
- Regex cleaning
- TF-IDF vectorization

**Accuracy:** ~98%

**Use Case:** Email / SMS spam detection
""")

Writing app.py


In [53]:
# Run Streamlit
from pyngrok import ngrok
import time
import os
ngrok.kill()
!streamlit run app.py --server.port 8501 --server.headless true &>/content/logs.txt &
time.sleep(5)
public_url = ngrok.connect(8501)
print("🌍 Open this URL:", public_url)

🌍 Open this URL: NgrokTunnel: "https://2ccd-34-11-104-138.ngrok-free.app" -> "http://localhost:8501"
